# Parte 1 · Recolección de titulares de noticias peruanas

Esta es la primera de las tres partes de la tarea: armar el conjunto de datos. En la Parte 2 los titulares se etiquetarán con un LLM y se hará el fine-tune de un Transformer ligero; en la Parte 3 se desplegará un dashboard.

Se trabaja con un solo medio, **El Comercio** (opción B del enunciado), y con cuatro categorías fijas: `politica`, `economia`, `deportes` y `ciencia_tecnologia`. Lo que no encaja en ninguna se descarta.

Pasos del notebook:

1. Revisar qué permite el `robots.txt`.
2. Descargar el archivo de cuatro secciones del diario.
3. Limpiar textos y fechas.
4. Quitar duplicados.
5. Mapear la sección original de cada nota a una de las cuatro categorías.
6. Verificar que cada URL responde.
7. Controlar que el total quede entre 350 y 400 titulares.
8. Revisar el resultado con tres gráficos.
9. Guardar `datos/noticias.csv`.

## 0. Librerías y reglas de cortesía

Todas las descargas pasan por la función `pedir`, que aplica las tres reglas del enunciado:

- **User-Agent descriptivo**: se presenta como el bot de un proyecto académico, sin datos personales.
- **Pausa de 1 a 2 segundos**, elegida al azar, después de cada pedido (también después de un reintento).
- **Máximo un reintento**, y solo cuando tiene sentido: error de red, error del servidor (5xx) o aviso de exceso de pedidos (429). Un 404 no se reintenta porque volvería a fallar.

Hace falta tener instalados `requests`, `beautifulsoup4`, `pandas`, `plotly` y `nbformat`. Plotly usa `nbformat` para dibujar los gráficos dentro del notebook.

In [ ]:
import json
import random
import re
import time
import xml.etree.ElementTree as ET
from email.utils import parsedate_to_datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import plotly.express as px
import requests
from bs4 import BeautifulSoup
from IPython.display import display

AGENTE = "TareaUniversitariaBot"
USER_AGENT = f"{AGENTE}/1.0 (proyecto academico de clasificacion de titulares; uso no comercial)"
BASE = "https://elcomercio.pe"
SECCIONES = ["politica", "economia", "deporte-total", "tecnologia"]
RSS = BASE + "/arc/outboundfeeds/rss/category/{seccion}/?outputType=xml"
CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
PAUSA = (1, 2)
MAX_REINTENTOS = 1
RUTA_CSV = Path("datos") / "noticias.csv"

sesion = requests.Session()
sesion.headers["User-Agent"] = USER_AGENT


def pedir(url, metodo="GET"):
    respuesta = None
    for _ in range(1 + MAX_REINTENTOS):
        try:
            respuesta = sesion.request(metodo, url, timeout=20, allow_redirects=True)
        except requests.RequestException:
            respuesta = None
        time.sleep(random.uniform(*PAUSA))
        if respuesta is not None and respuesta.status_code < 500 and respuesta.status_code != 429:
            break
    return respuesta

## 1. Qué permite el robots.txt de El Comercio

Antes de pedir cualquier página se revisa `https://elcomercio.pe/robots.txt`. Tiene dos bloques de reglas:

- **Para cualquier agente** (`User-agent: *`) prohíbe solo cuatro rutas: `/catalogo/`, `/buscar/` y dos tramos del archivo general por fechas, `/archivo/todas/19*` y `/archivo/todas/200*` (los años 1900 a 2009).
- **Para 27 scrapers con nombre propio** (HTTrack, wget, libwww, WebCopier, entre otros) prohíbe todo el sitio con `Disallow: /`. Nuestro User-Agent no figura en esa lista, así que le corresponde el primer bloque.

Las rutas que usa este notebook son el archivo por sección (`/archivo/politica/`, `/archivo/economia/`, `/archivo/deporte-total/` y `/archivo/tecnologia/`) y, en el paso de verificación, la página de cada nota. Ninguna cae en las rutas prohibidas: **están permitidas**. El archivo por sección no es el archivo general `/archivo/todas/`, que es el único que tiene tramos vedados.

La celda siguiente imprime el archivo y lo comprueba en código. No se usa `urllib.robotparser` de la librería estándar porque no entiende el comodín `*`: daría por permitida `/archivo/todas/2005-01-01/`, que está prohibida. En su lugar se aplica la regla del estándar RFC 9309: gana la regla más larga que coincide con la ruta y, si dos empatan, gana `Allow`. Las dos últimas filas de la tabla son controles que deben salir prohibidos.

In [ ]:
def reglas_robots(texto, agente):
    grupos, actual, leyendo_agentes = [], None, False
    for linea in texto.splitlines():
        linea = linea.split("#", 1)[0].strip()
        if ":" not in linea:
            continue
        campo, valor = (parte.strip() for parte in linea.split(":", 1))
        campo = campo.lower()
        if campo == "user-agent":
            if not leyendo_agentes:
                actual = {"agentes": [], "reglas": []}
                grupos.append(actual)
                leyendo_agentes = True
            actual["agentes"].append(valor.lower())
        elif campo in ("allow", "disallow") and actual:
            leyendo_agentes = False
            if valor:
                actual["reglas"].append((campo, valor))
    propios = [g for g in grupos if agente.lower() in g["agentes"]]
    elegidos = propios or [g for g in grupos if "*" in g["agentes"]]
    return [regla for g in elegidos for regla in g["reglas"]]


def permitido(ruta, reglas):
    coincidencias = []
    for tipo, patron in reglas:
        regex = re.escape(patron).replace(r"\*", ".*")
        if regex.endswith(r"\$"):
            regex = regex[:-2] + "$"
        if re.match(regex, ruta):
            coincidencias.append((len(patron), tipo == "allow"))
    return max(coincidencias)[1] if coincidencias else True


robots = pedir(f"{BASE}/robots.txt")
print(robots.text)

reglas = reglas_robots(robots.text, AGENTE)
rutas = [f"/archivo/{s}/" for s in SECCIONES] + ["/archivo/todas/2005-01-01/", "/buscar/peru/"]
pd.DataFrame({"ruta": rutas, "permitida": [permitido(ruta, reglas) for ruta in rutas]})

## 2. Descarga del archivo por sección

El Comercio tiene una página de archivo por sección (`/archivo/politica/`, por ejemplo) que lista sus 100 notas más recientes. Con cuatro pedidos, uno por sección, se cubren las cuatro categorías.

El listado visible trae titular, bajada y enlace, pero no la fecha. La misma página incluye en su HTML un bloque JSON (`Fusion.contentCache`) con esos datos de cada nota, más la fecha de publicación y la sección que le asignó el medio. Por eso se lee ese bloque en lugar de recorrer las etiquetas `<h2>`: viene en la misma respuesta y no requiere ningún pedido adicional.

**Plan B: RSS oficial.** Si el archivo de una sección devuelve error o deja de traer el JSON, el código usa el RSS oficial de esa sección y lo avisa en pantalla. En la prueba del 18 de octubre de 2026 el HTML respondió sin bloqueos y el RSS no hizo falta. Los RSS traen menos notas que el archivo (24 el de política y 79 el de deportes en la prueba), así que si se activara es probable que no se llegue a 350 titulares.

In [ ]:
def leer_archivo(html):
    inicio = html.index("Fusion.contentCache=") + len("Fusion.contentCache=")
    cache, _ = json.JSONDecoder().raw_decode(html, inicio)
    consultas = cache["story-feed-by-section"].values()
    notas = max(consultas, key=lambda c: len(c["data"]["content_elements"]))["data"]["content_elements"]
    filas = []
    for nota in notas:
        web = (nota.get("websites") or {}).get("elcomercio") or {}
        if not web.get("website_url"):
            continue
        filas.append({
            "titular": (nota.get("headlines") or {}).get("basic"),
            "bajada": (nota.get("subheadlines") or {}).get("basic"),
            "url": urljoin(BASE, web["website_url"]),
            "seccion_original": (web.get("website_section") or {}).get("path", "").strip("/"),
            "fecha": nota.get("display_date"),
        })
    return filas


def leer_rss(xml):
    filas = []
    for item in ET.fromstring(xml).iter("item"):
        url = item.findtext("link", "").strip()
        filas.append({
            "titular": item.findtext("title"),
            "bajada": BeautifulSoup(item.findtext("description", ""), "html.parser").get_text(" ", strip=True),
            "url": url,
            "seccion_original": "/".join(urlparse(url).path.strip("/").split("/")[:-1]),
            "fecha": parsedate_to_datetime(item.findtext("pubDate")).isoformat(),
        })
    return filas


filas, fuente = [], {}
for seccion in SECCIONES:
    respuesta = pedir(f"{BASE}/archivo/{seccion}/")
    try:
        if respuesta is None or respuesta.status_code != 200:
            raise ValueError(f"HTTP {getattr(respuesta, 'status_code', 'sin respuesta')}")
        nuevas, fuente[seccion] = leer_archivo(respuesta.text), "archivo HTML"
    except (ValueError, KeyError, TypeError) as error:
        print(f"{seccion}: el archivo HTML no sirvió ({error}); se usa el RSS oficial")
        nuevas, fuente[seccion] = leer_rss(pedir(RSS.format(seccion=seccion)).content), "RSS"
    filas += [{**fila, "seccion_archivo": seccion} for fila in nuevas]

df_crudo = pd.DataFrame(filas)
print(f"Titulares descargados: {len(df_crudo)}")
pd.DataFrame({"fuente": fuente, "titulares": df_crudo["seccion_archivo"].value_counts()})

## 3. Limpieza de textos y fechas

Dos arreglos antes de seguir:

- **Espacios.** Algunos titulares traen espacios dobles o sobrantes al inicio o al final (57 en la prueba). Se normalizan para que la deduplicación no trate como distintos dos textos iguales.
- **Fechas.** El JSON da la fecha en UTC y no siempre con el mismo formato: unas traen milisegundos y otras no. Por eso se lee con `format="ISO8601"`. Después se pasa a hora de Lima y se guarda solo el día, porque una nota publicada a las 10 p. m. del 16 en Lima figura en UTC como del 17.

La tabla muestra qué período cubre el archivo de cada sección.

In [ ]:
df = df_crudo.copy()
for columna in ["titular", "bajada"]:
    df[columna] = df[columna].fillna("").str.split().str.join(" ")
df = df[df["titular"] != ""]
df["fecha"] = pd.to_datetime(df["fecha"], utc=True, format="ISO8601").dt.tz_convert("America/Lima").dt.date
df["medio"] = "El Comercio"

print(f"Titulares sin bajada: {(df['bajada'] == '').sum()}")
print(f"Fechas: del {df['fecha'].min()} al {df['fecha'].max()}")
df.groupby("seccion_archivo")["fecha"].agg(desde="min", hasta="max")

## 6. Verificación de URLs

Se comprueba que cada URL responde con un pedido `HEAD`, que solo trae las cabeceras y no descarga la página. Antes de cada pedido se confirma con las reglas del `robots.txt` que la ruta está permitida. Las URLs que no devuelven 200 se muestran y se quitan.

Con la pausa obligatoria de 1 a 2 segundos, revisar unas 380 URLs toma alrededor de 10 minutos.

In [ ]:
print(f"Verificando {len(df)} URLs (unos {len(df) * 1.5 / 60:.0f} minutos)")
estados = []
for i, url in enumerate(df["url"], start=1):
    if not permitido(urlparse(url).path, reglas):
        estados.append("prohibida por robots.txt")
        continue
    respuesta = pedir(url, metodo="HEAD")
    estados.append(respuesta.status_code if respuesta is not None else "sin respuesta")
    if i % 50 == 0:
        print(f"  {i}/{len(df)}")

df["estado_http"] = estados
print(df["estado_http"].value_counts().to_string())
caidas = df[df["estado_http"] != 200]
print(f"URLs que no respondieron 200: {len(caidas)}")
display(caidas[["titular", "url", "estado_http"]])
df = df[df["estado_http"] == 200]

## 7. Control de volumen

El enunciado pide entre 350 y 400 titulares. Aquí se arma la tabla final con las siete columnas pedidas, en este orden, y se comprueba el rango.

In [ ]:
COLUMNAS = ["titular", "bajada", "medio", "seccion_original", "url", "fecha", "etiqueta"]
noticias = df[COLUMNAS].reset_index(drop=True)

cumple = 350 <= len(noticias) <= 400
print(f"Titulares finales: {len(noticias)} (rango pedido 350-400: {'cumple' if cumple else 'NO cumple'})")
print(noticias["etiqueta"].value_counts().reindex(CATEGORIAS).to_string())
noticias.head()

## 9. Guardar el CSV

Se guarda en `datos/noticias.csv`, una ruta relativa a la carpeta del notebook (`tarea_1/`), con `index=False` y `encoding="utf-8-sig"` para que Excel lea bien las tildes. Después se vuelve a leer el archivo para comprobar filas y columnas.

In [ ]:
RUTA_CSV.parent.mkdir(exist_ok=True)
noticias.to_csv(RUTA_CSV, index=False, encoding="utf-8-sig")

leido = pd.read_csv(RUTA_CSV, encoding="utf-8-sig")
print(f"Guardado {RUTA_CSV}: {leido.shape[0]} filas")
print(f"Columnas: {list(leido.columns)}")

## Limitaciones

- **Ventana de tiempo corta.** El archivo de cada sección solo entrega sus 100 notas más recientes, así que todos los titulares son de unos pocos días. En la corrida que generó `datos/noticias.csv` (18 de octubre de 2026, 384 titulares) las fechas van del 6 al 18 de octubre, 13 días en total, y el tramo cambia según la categoría: deportes cubre solo 5 días (del 14 al 18), economía 6 (del 12 al 17), política 7 (del 12 al 18) y ciencia y tecnología 12 (del 6 al 17). Los titulares reflejan la coyuntura de esos días: 16 tratan del debate de las facultades legislativas en el Senado y 15 del Torneo Clausura, entre la Liga 1 y la liga argentina. Un modelo entrenado con ellos puede aprender esos temas puntuales en vez de rasgos generales de cada categoría. Si se repite la recolección otro día, el rango cambia; el de cada corrida aparece en el paso 3 y en el gráfico 3.
- **Un solo medio.** Todo viene de El Comercio, con su estilo de redacción y su forma de organizar las secciones.
- **Etiqueta por sección.** La etiqueta dice dónde publicó el medio la nota, no de qué trata el titular, así que habrá casos dudosos, como una nota de economía sobre una ley del Congreso.
- **Descartes.** Las notas de opinión general, de ecdata y de otras secciones ajenas a las cuatro categorías se pierden, aunque varias traten esos temas.